# 01 — Data Preparation

**Project:** Stripped-context A vs trajectory B for next API-call prediction

This notebook:
1. Downloads API-Bank from HuggingFace
2. Extracts API names, excludes ToolSearcher (meta-API)
3. Builds 4 balanced API blocks via greedy bin-packing
4. Creates 80/20 train/eval splits per block
5. Formats data for Conditions A and B with the Llama 3.1 chat template
6. Saves preprocessed data as pickle

In [8]:
!pip install -q transformers datasets huggingface_hub numpy tqdm

In [9]:
import json
import os
import re
import random
import pickle
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Seed: {SEED}")

Seed: 42


## 1. Download API-Bank

In [10]:
data_files = [
    "training-data/lv1-train.json",
    "training-data/lv2-train.json",
    "training-data/lv3-train.json",
]

all_raw = []
for fname in data_files:
    path = hf_hub_download(
        repo_id="liminghao1630/API-Bank",
        filename=fname,
        repo_type="dataset",
    )
    with open(path) as f:
        entries = json.load(f)
    print(f"{fname}: {len(entries)} entries")
    all_raw.extend(entries)

print(f"\nTotal raw entries: {len(all_raw)}")
print(f"Keys: {list(all_raw[0].keys())}")

training-data/lv1-train.json: 6184 entries
training-data/lv2-train.json: 9279 entries
training-data/lv3-train.json: 1245 entries

Total raw entries: 16708
Keys: ['instruction', 'input', 'output']


## 2. Extract API Names & Filter

In [11]:
def extract_api_name(entry):
    # Extract primary API name from entry output or input
    text = entry.get('output', '') or ''
    match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', text)
    if match:
        return match.group(1)
    text = entry.get('input', '') or ''
    match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', text)
    if match:
        return match.group(1)
    return 'unknown'

for entry in all_raw:
    entry['api_name'] = extract_api_name(entry)

api_counts = defaultdict(int)
for entry in all_raw:
    api_counts[entry['api_name']] += 1

sorted_apis = sorted(api_counts.items(), key=lambda x: -x[1])
print(f"Unique APIs: {len(sorted_apis)}")
print(f"ToolSearcher entries: {api_counts.get('ToolSearcher', 0)}")
print(f"\nTop 10 APIs:")
for api, count in sorted_apis[:10]:
    print(f"  {api}: {count}")

Unique APIs: 1896
ToolSearcher entries: 6996

Top 10 APIs:
  ToolSearcher: 6996
  book_appointment: 94
  get_equipment_list: 91
  return_equipment: 78
  schedule_appointment: 71
  add_medication: 71
  cancel_appointment: 63
  get_claim_status: 58
  get_medications: 57
  submit_claim: 52


In [12]:
# Exclude ToolSearcher (meta-API) and unknown
filtered = [e for e in all_raw if e['api_name'] not in ('ToolSearcher', 'unknown')]
print(f"After excluding ToolSearcher and unknown: {len(filtered)} entries "
      f"(removed {len(all_raw) - len(filtered)})")

MIN_ENTRIES = 10
api_counts_filtered = defaultdict(int)
for entry in filtered:
    api_counts_filtered[entry['api_name']] += 1

valid_apis = sorted(
    [api for api, count in api_counts_filtered.items() if count >= MIN_ENTRIES]
)
valid_entries = [e for e in filtered if e['api_name'] in valid_apis]

print(f"APIs with >= {MIN_ENTRIES} entries: {len(valid_apis)}")
print(f"Entries from valid APIs: {len(valid_entries)}")

After excluding ToolSearcher and unknown: 9712 entries (removed 6996)
APIs with >= 10 entries: 185
Entries from valid APIs: 3474


## 3. Load Tokenizer

In [13]:
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")

Tokenizer loaded: meta-llama/Llama-3.1-8B-Instruct
Vocab size: 128256


## 4. Format Functions (Llama 3.1 Chat Format)

Both conditions predict the next API call with the same system prompt.
A strips prior API-Request/API-Response lines from the input.
B keeps the prior API trajectory.
This is stripped-context vs trajectory, not final-response-only vs trajectory.

In [14]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that can use tools. "
    "When you need to call an API, use the format: "
    "[ApiName(param1='value1', param2='value2')]. "
    "After receiving the API response, use it to formulate your answer."
)

def _strip_api_lines(text):
    # Remove API-Request, API-Response, and related lines
    lines = text.split('\n')
    out = []
    for line in lines:
        s = line.strip()
        if s.startswith('API-Request:') or s.startswith('API-Response:'):
            continue
        if 'Received API Response' in line or 'Generate API Request' in line:
            continue
        out.append(line)
    return '\n'.join(out).strip()

def build_chat_prompt(system_prompt, user_content, tokenizer):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_content},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def format_entry(entry, condition, tokenizer):
    # Format a single entry. Returns (full_text, prompt_token_len).
    inp = entry['input']
    out = entry.get('output', '')

    if condition == 'A':
        context = _strip_api_lines(inp)
    else:
        context = inp

    prompt = build_chat_prompt(SYSTEM_PROMPT, context, tokenizer)
    response = f"{out}{tokenizer.eos_token}"
    full_text = prompt + response
    prompt_len = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
    return full_text, prompt_len

# Quick test
sample = valid_entries[0]
for cond in ['A', 'B']:
    text, plen = format_entry(sample, cond, tokenizer)
    total = len(tokenizer.encode(text))
    print(f"Condition {cond}: {total} tokens (prompt: {plen}, response: {total - plen})")
print(f"\nSample target call: {sample.get('output', '')[:100]}")

Condition A: 657 tokens (prompt: 611, response: 46)
Condition B: 661 tokens (prompt: 615, response: 46)

Sample target call: API-Request: [schedule_appointment(patient_id='user's ID', doctor_id='Dr. Li's ID', appointment_time


## 5. Compute Token Counts & Greedy Bin-Packing

In [15]:
# Estimate token counts per API
api_stats = {}
for api in tqdm(valid_apis, desc="Token counting"):
    entries = [e for e in valid_entries if e['api_name'] == api]
    sample = entries[:min(20, len(entries))]
    tokens_a = [len(tokenizer.encode(format_entry(e, 'A', tokenizer)[0])) for e in sample]
    tokens_b = [len(tokenizer.encode(format_entry(e, 'B', tokenizer)[0])) for e in sample]
    api_stats[api] = {
        'count': len(entries),
        'avg_tokens_a': np.mean(tokens_a),
        'avg_tokens_b': np.mean(tokens_b),
        'est_total_b': np.mean(tokens_b) * len(entries),
    }

total_entries = sum(s['count'] for s in api_stats.values())
total_tokens_b = sum(s['est_total_b'] for s in api_stats.values())
print(f"Total entries: {total_entries}")
print(f"Estimated total tokens (B): {total_tokens_b:,.0f}")

Token counting:   0%|          | 0/185 [00:00<?, ?it/s]

Total entries: 3474
Estimated total tokens (B): 3,093,877


In [16]:
# Greedy bin-packing into 4 blocks
NUM_BLOCKS = 4
sorted_api_names = sorted(valid_apis, key=lambda x: -api_stats[x]['est_total_b'])

block_api_lists = [[] for _ in range(NUM_BLOCKS)]
block_token_totals = [0.0] * NUM_BLOCKS

for api in sorted_api_names:
    min_idx = int(np.argmin(block_token_totals))
    block_api_lists[min_idx].append(api)
    block_token_totals[min_idx] += api_stats[api]['est_total_b']

print("Block composition:")
for i in range(NUM_BLOCKS):
    n_apis = len(block_api_lists[i])
    n_entries = sum(api_stats[a]['count'] for a in block_api_lists[i])
    print(f"  D{i+1}: {n_apis} APIs, {n_entries} entries, ~{block_token_totals[i]:,.0f} tokens")

Block composition:
  D1: 46 APIs, 873 entries, ~771,966 tokens
  D2: 47 APIs, 875 entries, ~778,207 tokens
  D3: 46 APIs, 855 entries, ~771,883 tokens
  D4: 46 APIs, 871 entries, ~771,821 tokens


## 6. Build API Blocks with Train/Eval Splits

In [17]:
MAX_SEQ_LEN = 1024

domain_blocks = []
for i in range(NUM_BLOCKS):
    block_entries = [e for e in valid_entries if e['api_name'] in block_api_lists[i]]
    random.shuffle(block_entries)
    split_idx = int(len(block_entries) * 0.8)
    train_entries = block_entries[:split_idx]
    eval_entries = block_entries[split_idx:]
    domain_blocks.append({
        'block_id': i + 1,
        'apis': block_api_lists[i],
        'train_entries': train_entries,
        'eval_entries': eval_entries,
    })
    print(f"D{i+1}: {len(train_entries)} train, {len(eval_entries)} eval")

D1: 698 train, 175 eval
D2: 700 train, 175 eval
D3: 684 train, 171 eval
D4: 696 train, 175 eval


## 7. Format All Data & Compute Token Stats

In [18]:
blocks_data = []
BASE_EPOCHS = 3

for block in tqdm(domain_blocks, desc="Formatting"):
    bid = block['block_id']
    train_a, train_a_plens = [], []
    train_b, train_b_plens = [], []
    for e in block['train_entries']:
        ta, pa = format_entry(e, 'A', tokenizer)
        tb, pb = format_entry(e, 'B', tokenizer)
        train_a.append(ta); train_a_plens.append(pa)
        train_b.append(tb); train_b_plens.append(pb)

    eval_a, eval_a_plens = [], []
    eval_b, eval_b_plens = [], []
    for e in block['eval_entries']:
        ta, pa = format_entry(e, 'A', tokenizer)
        tb, pb = format_entry(e, 'B', tokenizer)
        eval_a.append(ta); eval_a_plens.append(pa)
        eval_b.append(tb); eval_b_plens.append(pb)

    tokens_a = sum(min(len(tokenizer.encode(t)), MAX_SEQ_LEN) for t in train_a)
    tokens_b = sum(min(len(tokenizer.encode(t)), MAX_SEQ_LEN) for t in train_b)
    ratio = tokens_b / tokens_a if tokens_a > 0 else 1.0

    blocks_data.append({
        'block_id': bid,
        'apis': block['apis'],
        'train_a': train_a, 'train_b': train_b,
        'train_a_prompt_lens': train_a_plens, 'train_b_prompt_lens': train_b_plens,
        'eval_a': eval_a, 'eval_b': eval_b,
        'eval_a_prompt_lens': eval_a_plens, 'eval_b_prompt_lens': eval_b_plens,
        'train_entries_raw': block['train_entries'],
        'eval_entries_raw': block['eval_entries'],
        'train_tokens_a': tokens_a, 'train_tokens_b': tokens_b,
        'token_ratio': ratio,
    })
    print(f"D{bid}: A={tokens_a:,} tok, B={tokens_b:,} tok, ratio={ratio:.2f}x")

Formatting:   0%|          | 0/4 [00:00<?, ?it/s]

D1: A=470,317 tok, B=586,733 tok, ratio=1.25x
D2: A=471,362 tok, B=587,908 tok, ratio=1.25x
D3: A=454,303 tok, B=569,510 tok, ratio=1.25x
D4: A=461,187 tok, B=580,163 tok, ratio=1.26x


## 8. Save Preprocessed Data

In [19]:
OUTPUT_DIR = "preprocessed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_data = {
    'blocks': blocks_data,
    'config': {
        'model_name': MODEL_NAME,
        'num_blocks': NUM_BLOCKS,
        'max_seq_len': MAX_SEQ_LEN,
        'base_epochs': BASE_EPOCHS,
        'seed': SEED,
        'min_entries': MIN_ENTRIES,
        'toolsearcher_excluded': True,
        'total_valid_apis': len(valid_apis),
        'total_valid_entries': len(valid_entries),
        'system_prompt': SYSTEM_PROMPT,
        'supported_conditions': ['A', 'B'],
    },
}

pkl_path = os.path.join(OUTPUT_DIR, 'preprocessed.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(save_data, f)

# Human-readable summary
summary = {
    'config': save_data['config'],
    'blocks': [{
        'block_id': b['block_id'],
        'num_apis': len(b['apis']),
        'apis': b['apis'][:10],
        'num_train': len(b['train_a']),
        'num_eval': len(b['eval_a']),
        'num_train_api_families': len(sorted({e['api_name'] for e in b['train_entries_raw']})),
        'num_eval_api_families': len(sorted({e['api_name'] for e in b['eval_entries_raw']})),
        'train_tokens_a': b['train_tokens_a'],
        'train_tokens_b': b['train_tokens_b'],
        'token_ratio': round(b['token_ratio'], 3),
    } for b in blocks_data],
}
json_path = os.path.join(OUTPUT_DIR, 'summary.json')
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)

pkl_size = os.path.getsize(pkl_path) / 1e6
print(f"\nSaved to {OUTPUT_DIR}/")
print(f"  preprocessed.pkl  ({pkl_size:.1f} MB)")
print(f"  summary.json")


Saved to preprocessed_data/
  preprocessed.pkl  (24.9 MB)
  summary.json
